In [2]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import csv 
from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
from keras import callbacks

from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout 
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, History 

from data_tools import load_preprocessed, dataPrep, nameModel

In [3]:
simPrefix = 'C:/Users/MOliv/Documents/PHY460/simFiles'
x, y = load_preprocessed(simPrefix, 'train')

Percentage of events with a NaN: 2.68


In [6]:
name = 'MaxPoolBatchNorm' 
i = 0
#make folder and make model save in that folder 
######while (os.path.exists('models/{}.h5'.format(name+str(i)))):
    i = i + 1 
name = name+str(i) 
numepochs = 100
prep = {'q':None, 't':False, 'normed':True, 'reco':'plane', 'cosz':False}

In [7]:
charge_input=keras.Input(shape=(10,10,2,))

conv1_layer = layers.Conv2D(64,kernel_size=3,padding='same',activation='relu')(charge_input)
batch1_layer = layers.BatchNormalization(axis=-1,momentum=0.99,epsilon=0.001,center=True,scale=True,beta_initializer="zeros",gamma_initializer="ones",moving_mean_initializer="zeros",moving_variance_initializer="ones",beta_regularizer=None,gamma_regularizer=None,beta_constraint=None,gamma_constraint=None)(conv1_layer)
conv2_layer = layers.Conv2D(32,kernel_size=3,padding='same',activation='relu')(batch1_layer)
batch2_layer = layers.BatchNormalization(axis=-1,momentum=0.99,epsilon=0.001,center=True,scale=True,beta_initializer="zeros",gamma_initializer="ones",moving_mean_initializer="zeros",moving_variance_initializer="ones",beta_regularizer=None,gamma_regularizer=None,beta_constraint=None,gamma_constraint=None)(conv2_layer)
conv3_layer = layers.Conv2D(16, kernel_size=3, padding='same',activation="relu")(batch2_layer) 
Mpool_layer = layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid", data_format=None) (conv3_layer)

flat_layer = layers.Flatten()(Mpool_layer)
zenith_input=keras.Input(shape=(1,))
concat_layer = layers.Concatenate()([flat_layer,zenith_input])

dense1_layer = layers.Dense(256,activation='relu')(concat_layer)
dense2_layer = layers.Dense(256,activation='relu')(dense1_layer)
dense3_layer = layers.Dense(256,activation="relu")(dense2_layer)

output = layers.Dense(1)(dense3_layer)

model = models.Model(inputs=[charge_input,zenith_input],outputs=output,name=name)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

In [8]:
model.summary()

Model: "MaxPoolBatchNorm0"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 10, 10, 2)]  0                                            
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 10, 10, 64)   1216        input_1[0][0]                    
__________________________________________________________________________________________________
batch_normalization (BatchNorma (None, 10, 10, 64)   256         conv2d[0][0]                     
__________________________________________________________________________________________________
conv2d_1 (Conv2D)               (None, 10, 10, 32)   18464       batch_normalization[0][0]        
__________________________________________________________________________________

In [9]:
len(np.sum(x,axis=(1,2,3)))

549773

In [10]:
th, _ = y['small_dir'].transpose()
len(th)

549773

In [11]:
x_i = dataPrep(x, y, **prep)

In [12]:
energy = y['energy']
comp = y['comp']
theta, phi = y['dir'].transpose()
nevents = len(energy)
trainCut = (np.random.uniform(size=nevents) < 0.85)
testCut = np.logical_not(trainCut)
temp_y = energy

In [13]:
np.count_nonzero(x_i[1]==None)

0

In [ ]:
csv_logger = CSVLogger('models/{}'.format(name))
early_stop = EarlyStopping(monitor="val_loss",min_delta=0,patience=10,verbose=0,mode="auto",baseline=None,restore_best_weights=False,) 
callbacks = [early_stop, csv_logger]
history = model.fit(
    {"charge":x_i[0],"zenith":x_i[1].reshape(-1,1)}, temp_y, epochs=numepochs,validation_split=0.15,callbacks=callbacks)

In [ ]:
np.save("MaxPoolBatchNorm.npy",prep)
#make a folder and make the file save to that folder 
#######model.save('models/%s.h5' % name)
f = open("results.txt", "a")
now = datetime.now()
f.write("{}\t{}\tepochs:{}\tloss:{},{}\n".format(
    now.strftime("%m/%d/%Y %H:%M:%S"),
    name,
    len(history.history['loss']),
    history.history['loss'][len(history.history['loss'])-1],
    history.history['val_loss'][len(history.history['loss'])-1]
))
f.close()